# Chapter 9.1: Deployment Options — Docker, Kubernetes, Bare Metal

This notebook covers production deployment strategies for vLLM and SGLang across
Docker, Kubernetes, and bare-metal environments. You will write real configuration
files, build Docker images, and set up monitoring infrastructure.

## Learning Objectives
- Build and customize Docker images for vLLM/SGLang serving
- Create docker-compose stacks with monitoring (Prometheus + Grafana)
- Write Kubernetes deployment manifests with GPU support
- Set up bare-metal systemd services
- Configure health checks and readiness probes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part9/chapter_9.1_deployment.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part9/chapter_9.1_deployment.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_deployment_architecture_comparison():
    """3-panel deployment architecture comparison: Docker, K8s, Bare Metal."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    titles = ['Docker (Single Container)', 'Kubernetes (Pods + Service + HPA)', 'Bare Metal (systemd + nginx)']
    colors = {'main': '#4A90D9', 'support': '#27AE60', 'infra': '#F39C12', 'external': '#8E44AD'}

    # --- Docker ---
    ax = axes[0]
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
    ax.set_title(titles[0], fontsize=11, fontweight='bold', color='#4A90D9')
    docker_components = [
        (1, 7.5, 8, 1.5, 'Docker Host', '#E3F2FD', '#4A90D9'),
        (2, 5.0, 6, 2.0, 'vLLM Container\n(GPU runtime + model)', '#4A90D9', 'white'),
        (2, 2.5, 6, 1.8, 'Volumes\n(model cache + /dev/shm)', '#F39C12', 'white'),
        (2, 0.5, 6, 1.5, 'NVIDIA Container Toolkit', '#27AE60', 'white'),
    ]
    for x, y, w, h, label, bg, fg in docker_components:
        rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                              facecolor=bg, edgecolor='#333', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=9,
                fontweight='bold', color=fg)

    # --- Kubernetes ---
    ax = axes[1]
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
    ax.set_title(titles[1], fontsize=11, fontweight='bold', color='#27AE60')
    k8s_components = [
        (0.5, 8.0, 9, 1.2, 'K8s Service (LoadBalancer)', '#8E44AD', 'white'),
        (0.5, 5.5, 4, 2.0, 'Pod 1\nvLLM + GPU', '#4A90D9', 'white'),
        (5.5, 5.5, 4, 2.0, 'Pod 2\nvLLM + GPU', '#4A90D9', 'white'),
        (0.5, 3.5, 9, 1.5, 'HPA (Auto-scaling on GPU metrics)', '#F39C12', 'white'),
        (0.5, 1.5, 4, 1.5, 'PVC\n(Model Cache)', '#27AE60', 'white'),
        (5.5, 1.5, 4, 1.5, 'ConfigMap\n+ Secrets', '#27AE60', 'white'),
    ]
    for x, y, w, h, label, bg, fg in k8s_components:
        rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                              facecolor=bg, edgecolor='#333', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8,
                fontweight='bold', color=fg)

    # --- Bare Metal ---
    ax = axes[2]
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
    ax.set_title(titles[2], fontsize=11, fontweight='bold', color='#F39C12')
    bare_components = [
        (1, 8.0, 8, 1.2, 'nginx (reverse proxy + LB)', '#8E44AD', 'white'),
        (1, 5.5, 8, 2.0, 'systemd Service\nvLLM (auto-restart)', '#4A90D9', 'white'),
        (1, 3.5, 8, 1.5, 'CUDA Driver + GPU Direct', '#27AE60', 'white'),
        (1, 1.5, 3.5, 1.5, 'journald\n(logging)', '#F39C12', 'white'),
        (5.5, 1.5, 3.5, 1.5, '/data/models\n(local SSD)', '#F39C12', 'white'),
    ]
    for x, y, w, h, label, bg, fg in bare_components:
        rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                              facecolor=bg, edgecolor='#333', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8,
                fontweight='bold', color=fg)

    plt.tight_layout()
    plt.savefig('deployment_architecture_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_deployment_architecture_comparison()

## 1. Prerequisites and Environment Setup

In [ ]:
import os
import json
import subprocess
import textwrap
from pathlib import Path

# Working directory for deployment configs
DEPLOY_DIR = Path("./deployment_configs")
DEPLOY_DIR.mkdir(exist_ok=True)

def write_config(filename, content, subdir=None):
    """Write a config file and display it."""
    target_dir = DEPLOY_DIR / subdir if subdir else DEPLOY_DIR
    target_dir.mkdir(parents=True, exist_ok=True)
    path = target_dir / filename
    path.write_text(textwrap.dedent(content).strip() + "\n")
    print(f"Written: {path}")
    print(f"{'='*60}")
    print(path.read_text())
    return path

print("Setup complete. Deployment configs will be written to:", DEPLOY_DIR.resolve())

## 2. Demo: Build a vLLM Docker Image from Scratch

We will create a production-grade Dockerfile for vLLM with multi-stage build,
proper caching, and security best practices.

In [ ]:
# --- Dockerfile for vLLM production deployment ---
vllm_dockerfile = """\
# Stage 1: Builder - install dependencies
FROM nvidia/cuda:12.4.1-devel-ubuntu22.04 AS builder

ENV DEBIAN_FRONTEND=noninteractive
RUN apt-get update && apt-get install -y --no-install-recommends \\
    python3.11 python3.11-venv python3-pip git curl && \\
    rm -rf /var/lib/apt/lists/*

RUN python3.11 -m venv /opt/vllm-env
ENV PATH="/opt/vllm-env/bin:$PATH"

# Install vLLM with pinned version for reproducibility
RUN pip install --no-cache-dir \\
    vllm==0.6.6 \\
    prometheus-client \\
    uvloop

# Stage 2: Runtime - minimal image
FROM nvidia/cuda:12.4.1-runtime-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive
RUN apt-get update && apt-get install -y --no-install-recommends \\
    python3.11 python3.11-venv curl && \\
    rm -rf /var/lib/apt/lists/*

# Copy virtualenv from builder
COPY --from=builder /opt/vllm-env /opt/vllm-env
ENV PATH="/opt/vllm-env/bin:$PATH"

# Non-root user for security
RUN groupadd -r vllm && useradd -r -g vllm -m vllm

# Model cache directory
RUN mkdir -p /models && chown vllm:vllm /models
ENV HF_HOME=/models

USER vllm
WORKDIR /app

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

EXPOSE 8000

ENTRYPOINT ["python3", "-m", "vllm.entrypoints.openai.api_server"]
CMD ["--model", "facebook/opt-125m", "--host", "0.0.0.0", "--port", "8000"]
"""

write_config("Dockerfile.vllm", vllm_dockerfile, subdir="docker")

In [ ]:
# --- .dockerignore for clean builds ---
dockerignore = """\
__pycache__
*.pyc
.git
.env
*.egg-info
dist
build
notebooks
.vscode
*.log
"""

write_config(".dockerignore", dockerignore, subdir="docker")

# --- Build script ---
build_script = """\
#!/bin/bash
set -euo pipefail

IMAGE_NAME="vllm-serving"
IMAGE_TAG="${1:-latest}"

echo "Building vLLM Docker image: ${IMAGE_NAME}:${IMAGE_TAG}"

docker build \\
    -f Dockerfile.vllm \\
    -t "${IMAGE_NAME}:${IMAGE_TAG}" \\
    --build-arg BUILDKIT_INLINE_CACHE=1 \\
    .

echo "Image built successfully."
docker images "${IMAGE_NAME}:${IMAGE_TAG}"

# Quick sanity check (no GPU needed)
echo "Running sanity check..."
docker run --rm "${IMAGE_NAME}:${IMAGE_TAG}" python3 -c "import vllm; print(f'vLLM version: {vllm.__version__}')"
"""

write_config("build.sh", build_script, subdir="docker")

## 3. Demo: Docker-Compose with Prometheus + Grafana

A complete monitoring stack: vLLM + Prometheus for metrics collection + Grafana for dashboards.

In [ ]:
# --- docker-compose.yml ---
docker_compose = """\
version: '3.8'

services:
  vllm:
    image: vllm/vllm-openai:latest
    container_name: vllm-server
    runtime: nvidia
    environment:
      - NVIDIA_VISIBLE_DEVICES=all
      - VLLM_USAGE_STATS=0
    ports:
      - "8000:8000"
    volumes:
      - model-cache:/root/.cache/huggingface
    command:
      - "--model"
      - "facebook/opt-1.3b"
      - "--host"
      - "0.0.0.0"
      - "--port"
      - "8000"
      - "--max-model-len"
      - "2048"
      - "--gpu-memory-utilization"
      - "0.9"
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 5
      start_period: 120s
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    restart: unless-stopped

  prometheus:
    image: prom/prometheus:v2.50.1
    container_name: prometheus
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml:ro
      - ./alert_rules.yml:/etc/prometheus/alert_rules.yml:ro
      - prometheus-data:/prometheus
    command:
      - '--config.file=/etc/prometheus/prometheus.yml'
      - '--storage.tsdb.retention.time=30d'
    depends_on:
      vllm:
        condition: service_healthy
    restart: unless-stopped

  grafana:
    image: grafana/grafana:10.3.1
    container_name: grafana
    ports:
      - "3000:3000"
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=admin
      - GF_USERS_ALLOW_SIGN_UP=false
    volumes:
      - grafana-data:/var/lib/grafana
      - ./grafana/provisioning:/etc/grafana/provisioning:ro
    depends_on:
      - prometheus
    restart: unless-stopped

volumes:
  model-cache:
  prometheus-data:
  grafana-data:
"""

write_config("docker-compose.yml", docker_compose, subdir="docker")

In [ ]:
# --- Prometheus configuration ---
prometheus_config = """\
global:
  scrape_interval: 15s
  evaluation_interval: 15s

rule_files:
  - "alert_rules.yml"

scrape_configs:
  - job_name: 'vllm'
    static_configs:
      - targets: ['vllm:8000']
    metrics_path: '/metrics'
    scrape_interval: 5s

  - job_name: 'prometheus'
    static_configs:
      - targets: ['localhost:9090']
"""

write_config("prometheus.yml", prometheus_config, subdir="docker")

# --- Alert rules ---
alert_rules = """\
groups:
  - name: vllm_alerts
    rules:
      - alert: HighP99Latency
        expr: histogram_quantile(0.99, rate(vllm:request_latency_seconds_bucket[5m])) > 10
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "P99 latency exceeds 10s"
          description: "vLLM P99 request latency is {{ $value }}s"

      - alert: HighGPUMemoryUsage
        expr: vllm:gpu_cache_usage_perc > 0.95
        for: 2m
        labels:
          severity: critical
        annotations:
          summary: "GPU KV-cache usage above 95%"

      - alert: ServiceDown
        expr: up{job="vllm"} == 0
        for: 1m
        labels:
          severity: critical
        annotations:
          summary: "vLLM service is down"
"""

write_config("alert_rules.yml", alert_rules, subdir="docker")

In [ ]:
# --- Grafana datasource provisioning ---
grafana_datasource = """\
apiVersion: 1

datasources:
  - name: Prometheus
    type: prometheus
    access: proxy
    url: http://prometheus:9090
    isDefault: true
    editable: false
"""

write_config("datasources.yml", grafana_datasource, subdir="docker/grafana/provisioning/datasources")
print("\nGrafana will auto-discover Prometheus at startup.")

## 4. Demo: Kubernetes Deployment for vLLM

Production Kubernetes manifests with GPU scheduling, resource limits, probes, and PDB.

In [ ]:
# --- Kubernetes Namespace and Deployment ---
k8s_namespace = """\
apiVersion: v1
kind: Namespace
metadata:
  name: llm-serving
  labels:
    app.kubernetes.io/part-of: llm-platform
"""

write_config("namespace.yaml", k8s_namespace, subdir="k8s")

k8s_deployment = """\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm-server
  namespace: llm-serving
  labels:
    app: vllm-server
    version: v1
spec:
  replicas: 2
  selector:
    matchLabels:
      app: vllm-server
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0
  template:
    metadata:
      labels:
        app: vllm-server
        version: v1
      annotations:
        prometheus.io/scrape: "true"
        prometheus.io/port: "8000"
        prometheus.io/path: "/metrics"
    spec:
      nodeSelector:
        nvidia.com/gpu.present: "true"
      tolerations:
        - key: nvidia.com/gpu
          operator: Exists
          effect: NoSchedule
      containers:
        - name: vllm
          image: vllm/vllm-openai:latest
          args:
            - "--model"
            - "meta-llama/Llama-3.1-8B-Instruct"
            - "--host"
            - "0.0.0.0"
            - "--port"
            - "8000"
            - "--tensor-parallel-size"
            - "1"
            - "--max-model-len"
            - "4096"
            - "--gpu-memory-utilization"
            - "0.9"
            - "--enable-prefix-caching"
          ports:
            - containerPort: 8000
              name: http
              protocol: TCP
          env:
            - name: HUGGING_FACE_HUB_TOKEN
              valueFrom:
                secretKeyRef:
                  name: hf-secret
                  key: token
            - name: VLLM_USAGE_STATS
              value: "0"
          resources:
            requests:
              cpu: "4"
              memory: "32Gi"
              nvidia.com/gpu: "1"
            limits:
              cpu: "8"
              memory: "64Gi"
              nvidia.com/gpu: "1"
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 120
            periodSeconds: 10
            timeoutSeconds: 5
            failureThreshold: 3
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 180
            periodSeconds: 30
            timeoutSeconds: 10
            failureThreshold: 5
          startupProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 30
            periodSeconds: 10
            failureThreshold: 30
          volumeMounts:
            - name: model-cache
              mountPath: /root/.cache/huggingface
            - name: shm
              mountPath: /dev/shm
      volumes:
        - name: model-cache
          persistentVolumeClaim:
            claimName: model-cache-pvc
        - name: shm
          emptyDir:
            medium: Memory
            sizeLimit: 16Gi
"""

write_config("deployment.yaml", k8s_deployment, subdir="k8s")

In [ ]:
# --- Service and PodDisruptionBudget ---
k8s_service = """\
apiVersion: v1
kind: Service
metadata:
  name: vllm-service
  namespace: llm-serving
  labels:
    app: vllm-server
spec:
  type: ClusterIP
  ports:
    - port: 80
      targetPort: 8000
      protocol: TCP
      name: http
  selector:
    app: vllm-server
---
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: vllm-pdb
  namespace: llm-serving
spec:
  minAvailable: 1
  selector:
    matchLabels:
      app: vllm-server
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: model-cache-pvc
  namespace: llm-serving
spec:
  accessModes:
    - ReadWriteMany
  resources:
    requests:
      storage: 100Gi
  storageClassName: fast-ssd
"""

write_config("service.yaml", k8s_service, subdir="k8s")

## 5. Demo: Bare Metal systemd Service Setup

In [ ]:
# --- systemd unit file for vLLM ---
systemd_unit = """\
[Unit]
Description=vLLM OpenAI-compatible API Server
After=network-online.target nvidia-persistenced.service
Wants=network-online.target
Documentation=https://docs.vllm.ai

[Service]
Type=simple
User=vllm
Group=vllm
WorkingDirectory=/opt/vllm

# Environment
Environment=CUDA_VISIBLE_DEVICES=0
Environment=HF_HOME=/data/models
Environment=VLLM_USAGE_STATS=0
EnvironmentFile=-/etc/vllm/vllm.env

# Main process
ExecStart=/opt/vllm/venv/bin/python -m vllm.entrypoints.openai.api_server \\
    --model meta-llama/Llama-3.1-8B-Instruct \\
    --host 0.0.0.0 \\
    --port 8000 \\
    --max-model-len 4096 \\
    --gpu-memory-utilization 0.9 \\
    --enable-prefix-caching

# Restart policy
Restart=on-failure
RestartSec=30
StartLimitIntervalSec=300
StartLimitBurst=3

# Resource limits
LimitNOFILE=65536
LimitMEMLOCK=infinity

# Security hardening
NoNewPrivileges=true
ProtectSystem=strict
ReadWritePaths=/data/models /var/log/vllm
PrivateTmp=true

# Logging
StandardOutput=journal
StandardError=journal
SyslogIdentifier=vllm

[Install]
WantedBy=multi-user.target
"""

write_config("vllm.service", systemd_unit, subdir="systemd")

In [ ]:
# --- Bare metal installation script ---
install_script = """\
#!/bin/bash
set -euo pipefail

echo "=== vLLM Bare Metal Installation ==="

# 1. Create service user
sudo useradd -r -m -d /opt/vllm -s /bin/bash vllm 2>/dev/null || true
sudo usermod -aG video vllm  # GPU access

# 2. Create directories
sudo mkdir -p /opt/vllm /data/models /etc/vllm /var/log/vllm
sudo chown -R vllm:vllm /opt/vllm /data/models /var/log/vllm

# 3. Create virtual environment
sudo -u vllm python3.11 -m venv /opt/vllm/venv
sudo -u vllm /opt/vllm/venv/bin/pip install --upgrade pip
sudo -u vllm /opt/vllm/venv/bin/pip install vllm

# 4. Create environment file
cat > /etc/vllm/vllm.env << 'ENVEOF'
HUGGING_FACE_HUB_TOKEN=your_token_here
CUDA_VISIBLE_DEVICES=0
VLLM_USAGE_STATS=0
ENVEOF
chmod 600 /etc/vllm/vllm.env

# 5. Install systemd unit
sudo cp vllm.service /etc/systemd/system/
sudo systemctl daemon-reload
sudo systemctl enable vllm

echo "=== Installation complete ==="
echo "Start with: sudo systemctl start vllm"
echo "Check logs: journalctl -u vllm -f"
"""

write_config("install.sh", install_script, subdir="systemd")

## 6. SGLang Deployment Configs

Equivalent deployment configurations for SGLang.

In [ ]:
# --- SGLang Dockerfile ---
sglang_dockerfile = """\
FROM nvidia/cuda:12.4.1-runtime-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive
RUN apt-get update && apt-get install -y --no-install-recommends \\
    python3.11 python3.11-venv python3-pip curl && \\
    rm -rf /var/lib/apt/lists/*

RUN python3.11 -m venv /opt/sglang-env
ENV PATH="/opt/sglang-env/bin:$PATH"

RUN pip install --no-cache-dir "sglang[all]" 

RUN groupadd -r sglang && useradd -r -g sglang -m sglang
RUN mkdir -p /models && chown sglang:sglang /models
ENV HF_HOME=/models

USER sglang
WORKDIR /app

HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=3 \\
    CMD curl -f http://localhost:30000/health || exit 1

EXPOSE 30000

ENTRYPOINT ["python3", "-m", "sglang.launch_server"]
CMD ["--model-path", "meta-llama/Llama-3.1-8B-Instruct", "--host", "0.0.0.0", "--port", "30000"]
"""

write_config("Dockerfile.sglang", sglang_dockerfile, subdir="docker")

# --- SGLang K8s Deployment ---
sglang_k8s = """\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: sglang-server
  namespace: llm-serving
spec:
  replicas: 2
  selector:
    matchLabels:
      app: sglang-server
  template:
    metadata:
      labels:
        app: sglang-server
    spec:
      nodeSelector:
        nvidia.com/gpu.present: "true"
      containers:
        - name: sglang
          image: lmsysorg/sglang:latest
          args:
            - "--model-path"
            - "meta-llama/Llama-3.1-8B-Instruct"
            - "--host"
            - "0.0.0.0"
            - "--port"
            - "30000"
            - "--tp"
            - "1"
          ports:
            - containerPort: 30000
          resources:
            limits:
              nvidia.com/gpu: "1"
              memory: "64Gi"
          readinessProbe:
            httpGet:
              path: /health
              port: 30000
            initialDelaySeconds: 120
            periodSeconds: 10
          volumeMounts:
            - name: shm
              mountPath: /dev/shm
      volumes:
        - name: shm
          emptyDir:
            medium: Memory
            sizeLimit: 16Gi
"""

write_config("sglang-deployment.yaml", sglang_k8s, subdir="k8s")

## 7. Health Check and Readiness Probe Scripts

In [ ]:
import time
import json
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class HealthCheckResult:
    """Result of a health check."""
    healthy: bool
    endpoint: str
    response_time_ms: float
    status_code: Optional[int] = None
    details: dict = field(default_factory=dict)
    error: Optional[str] = None

class LLMHealthChecker:
    """Comprehensive health checker for LLM serving endpoints."""
    
    def __init__(self, base_url: str, timeout: float = 10.0):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
    
    def check_health(self) -> HealthCheckResult:
        """Basic health check - is the server responding?"""
        import urllib.request
        start = time.time()
        try:
            req = urllib.request.Request(f"{self.base_url}/health")
            with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                elapsed = (time.time() - start) * 1000
                return HealthCheckResult(
                    healthy=resp.status == 200,
                    endpoint="/health",
                    response_time_ms=elapsed,
                    status_code=resp.status
                )
        except Exception as e:
            elapsed = (time.time() - start) * 1000
            return HealthCheckResult(
                healthy=False,
                endpoint="/health",
                response_time_ms=elapsed,
                error=str(e)
            )
    
    def check_model_loaded(self) -> HealthCheckResult:
        """Check that models are loaded and available."""
        import urllib.request
        start = time.time()
        try:
            req = urllib.request.Request(f"{self.base_url}/v1/models")
            with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                data = json.loads(resp.read())
                elapsed = (time.time() - start) * 1000
                models = [m["id"] for m in data.get("data", [])]
                return HealthCheckResult(
                    healthy=len(models) > 0,
                    endpoint="/v1/models",
                    response_time_ms=elapsed,
                    status_code=resp.status,
                    details={"models": models}
                )
        except Exception as e:
            elapsed = (time.time() - start) * 1000
            return HealthCheckResult(
                healthy=False,
                endpoint="/v1/models",
                response_time_ms=elapsed,
                error=str(e)
            )
    
    def check_inference(self, model: str = None) -> HealthCheckResult:
        """Deep health check - can the model actually generate?"""
        import urllib.request
        start = time.time()
        try:
            payload = json.dumps({
                "model": model or "default",
                "messages": [{"role": "user", "content": "Say OK"}],
                "max_tokens": 5,
                "temperature": 0
            }).encode()
            req = urllib.request.Request(
                f"{self.base_url}/v1/chat/completions",
                data=payload,
                headers={"Content-Type": "application/json"}
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read())
                elapsed = (time.time() - start) * 1000
                return HealthCheckResult(
                    healthy=True,
                    endpoint="/v1/chat/completions",
                    response_time_ms=elapsed,
                    status_code=resp.status,
                    details={"response": data["choices"][0]["message"]["content"]}
                )
        except Exception as e:
            elapsed = (time.time() - start) * 1000
            return HealthCheckResult(
                healthy=False,
                endpoint="/v1/chat/completions",
                response_time_ms=elapsed,
                error=str(e)
            )
    
    def full_check(self, model: str = None) -> dict:
        """Run all health checks."""
        results = {
            "health": self.check_health(),
            "models": self.check_model_loaded(),
            "inference": self.check_inference(model)
        }
        overall = all(r.healthy for r in results.values())
        print(f"\n{'='*50}")
        print(f"Health Check Report - {self.base_url}")
        print(f"{'='*50}")
        for name, result in results.items():
            status = "PASS" if result.healthy else "FAIL"
            print(f"  [{status}] {name:12s} | {result.response_time_ms:7.1f}ms | {result.endpoint}")
            if result.error:
                print(f"         Error: {result.error}")
            if result.details:
                print(f"         Details: {result.details}")
        print(f"\n  Overall: {'HEALTHY' if overall else 'UNHEALTHY'}")
        return results

# Demo usage (will fail without a running server, but shows the API)
checker = LLMHealthChecker("http://localhost:8000")
print("Health checker created. Usage:")
print("  checker.check_health()       - Basic liveness check")
print("  checker.check_model_loaded() - Verify models are available")
print("  checker.check_inference()    - End-to-end inference test")
print("  checker.full_check()         - All checks with report")
print("\n# Example: checker.full_check(model='meta-llama/Llama-3.1-8B-Instruct')")

In [ ]:
# --- Readiness gate script for K8s init containers ---
readiness_script = """\
#!/bin/bash
# Wait for vLLM to be ready before allowing traffic
set -euo pipefail

VLLM_URL="${VLLM_URL:-http://localhost:8000}"
MAX_WAIT="${MAX_WAIT:-600}"  # 10 minutes max
CHECK_INTERVAL="${CHECK_INTERVAL:-5}"

echo "Waiting for vLLM at ${VLLM_URL} (timeout: ${MAX_WAIT}s)..."

elapsed=0
while [ $elapsed -lt $MAX_WAIT ]; do
    if curl -sf "${VLLM_URL}/health" > /dev/null 2>&1; then
        # Health endpoint responded. Now check models are loaded.
        model_count=$(curl -sf "${VLLM_URL}/v1/models" | python3 -c \
            "import sys,json; print(len(json.load(sys.stdin).get('data',[])))" 2>/dev/null || echo 0)
        
        if [ "$model_count" -gt 0 ]; then
            echo "vLLM is ready! (${model_count} model(s) loaded, ${elapsed}s elapsed)"
            exit 0
        fi
        echo "  Health OK, but no models loaded yet (${elapsed}s)..."
    else
        echo "  Not ready yet (${elapsed}s)..."
    fi
    sleep $CHECK_INTERVAL
    elapsed=$((elapsed + CHECK_INTERVAL))
done

echo "TIMEOUT: vLLM did not become ready within ${MAX_WAIT}s"
exit 1
"""

write_config("wait-for-ready.sh", readiness_script, subdir="scripts")

## 8. Deployment Comparison Summary

In [ ]:
comparison = {
    "Method": ["Docker", "Docker Compose", "Kubernetes", "Bare Metal"],
    "Setup Complexity": ["Low", "Low-Medium", "High", "Medium"],
    "Auto-scaling": ["Manual", "Manual", "Built-in (HPA)", "Manual/Custom"],
    "Load Balancing": ["External", "External/Nginx", "Built-in (Service)", "External"],
    "Health Checks": ["HEALTHCHECK", "HEALTHCHECK", "Probes (L/R/S)", "systemd/Custom"],
    "GPU Support": ["--gpus / runtime", "deploy.resources", "nvidia device plugin", "Native"],
    "Multi-node": ["Swarm (limited)", "Swarm (limited)", "Native", "Manual"],
    "Best For": ["Dev/Test", "Small prod", "Large prod", "Single server"],
}

# Pretty print as table
col_widths = {k: max(len(k), max(len(str(v)) for v in vals)) for k, vals in comparison.items()}
header = " | ".join(f"{k:{col_widths[k]}}" for k in comparison)
separator = "-|-".join("-" * col_widths[k] for k in comparison)
print(header)
print(separator)
for i in range(len(comparison["Method"])):
    row = " | ".join(f"{comparison[k][i]:{col_widths[k]}}" for k in comparison)
    print(row)

---
## Hands-on Assignments

### Assignment 1: Deploy vLLM in Docker with GPU Passthrough

Create a complete Docker deployment that:
1. Uses GPU passthrough with the NVIDIA Container Toolkit
2. Mounts a local model directory
3. Sets proper resource limits
4. Includes a health check

In [ ]:
# ASSIGNMENT 1: Complete the Docker run command and test script

def generate_docker_run_command(
    model_name: str,
    gpu_ids: list[int],
    port: int = 8000,
    max_model_len: int = 4096,
    gpu_memory_util: float = 0.9,
    model_cache_dir: str = "/data/models",
    container_name: str = "vllm-prod",
) -> str:
    """Generate a docker run command for vLLM with GPU passthrough.
    
    Requirements:
    - Use nvidia runtime
    - Set NVIDIA_VISIBLE_DEVICES to the specified GPU IDs
    - Mount model_cache_dir to /root/.cache/huggingface
    - Mount /dev/shm with 16GB for shared memory
    - Set memory limit to 64GB
    - Add health check
    - Set restart policy to unless-stopped
    
    Returns:
        Complete docker run command as a string.
    """
    # TODO: Build the docker run command
    # Hint: Use --runtime=nvidia or --gpus flag
    # Hint: Use -e NVIDIA_VISIBLE_DEVICES=...
    # Hint: Use -v for volume mounts
    # Hint: Use --health-cmd for health checks
    
    gpu_str = ",".join(str(g) for g in gpu_ids)
    
    cmd_parts = [
        "docker run -d",
        f"--name {container_name}",
        # ADD YOUR CODE HERE: GPU runtime
        # ADD YOUR CODE HERE: GPU device selection
        # ADD YOUR CODE HERE: Port mapping
        # ADD YOUR CODE HERE: Volume mounts (model cache + /dev/shm)
        # ADD YOUR CODE HERE: Memory limit
        # ADD YOUR CODE HERE: Restart policy
        # ADD YOUR CODE HERE: Health check
        "vllm/vllm-openai:latest",
        f"--model {model_name}",
        f"--max-model-len {max_model_len}",
        f"--gpu-memory-utilization {gpu_memory_util}",
        "--host 0.0.0.0",
        f"--port 8000",
    ]
    
    return " \\\n    ".join(cmd_parts)


# Test your implementation
cmd = generate_docker_run_command(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    gpu_ids=[0],
    port=8000,
)
print("Generated docker run command:")
print(cmd)

# Expected output should include:
# --runtime=nvidia or --gpus
# -e NVIDIA_VISIBLE_DEVICES=0
# -v /data/models:/root/.cache/huggingface
# --shm-size=16g or --tmpfs /dev/shm:size=16g
# --memory=64g
# --restart=unless-stopped
# --health-cmd

In [ ]:
# ASSIGNMENT 1 - SOLUTION

def generate_docker_run_command_solution(
    model_name: str,
    gpu_ids: list[int],
    port: int = 8000,
    max_model_len: int = 4096,
    gpu_memory_util: float = 0.9,
    model_cache_dir: str = "/data/models",
    container_name: str = "vllm-prod",
) -> str:
    gpu_str = ",".join(str(g) for g in gpu_ids)
    
    cmd_parts = [
        "docker run -d",
        f"--name {container_name}",
        f"--runtime=nvidia",
        f"-e NVIDIA_VISIBLE_DEVICES={gpu_str}",
        f"-p {port}:8000",
        f"-v {model_cache_dir}:/root/.cache/huggingface",
        f"--shm-size=16g",
        f"--memory=64g",
        f"--restart=unless-stopped",
        '--health-cmd="curl -f http://localhost:8000/health || exit 1"',
        "--health-interval=30s",
        "--health-timeout=10s",
        "--health-start-period=120s",
        "--health-retries=3",
        "vllm/vllm-openai:latest",
        f"--model {model_name}",
        f"--max-model-len {max_model_len}",
        f"--gpu-memory-utilization {gpu_memory_util}",
        "--host 0.0.0.0",
        "--port 8000",
    ]
    
    return " \\\n    ".join(cmd_parts)

cmd = generate_docker_run_command_solution(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    gpu_ids=[0, 1],
    port=8080,
)
print("Solution docker run command:")
print(cmd)

### Assignment 2: Create a Kubernetes HPA Configuration

Write a HorizontalPodAutoscaler that scales vLLM pods based on custom GPU metrics.

In [ ]:
# ASSIGNMENT 2: Create a K8s HPA config for vLLM

def generate_hpa_config(
    name: str = "vllm-hpa",
    namespace: str = "llm-serving",
    target_deployment: str = "vllm-server",
    min_replicas: int = 1,
    max_replicas: int = 8,
    cpu_target_percent: int = 70,
    gpu_utilization_target: int = 80,
    queue_depth_target: int = 50,
) -> str:
    """Generate a Kubernetes HPA YAML for vLLM auto-scaling.
    
    The HPA should:
    1. Scale based on CPU utilization (resource metric)
    2. Scale based on GPU utilization (custom pod metric)
    3. Scale based on request queue depth (custom pod metric)  
    4. Have scale-down stabilization of 300s to prevent flapping
    5. Have scale-up stabilization of 60s for fast response
    
    Returns:
        Valid Kubernetes HPA YAML as a string.
    """
    # TODO: Write the HPA YAML
    # Hint: Use apiVersion: autoscaling/v2
    # Hint: Use metrics[].type: Resource for CPU
    # Hint: Use metrics[].type: Pods for custom metrics
    # Hint: Use behavior.scaleDown.stabilizationWindowSeconds
    
    hpa_yaml = f"""\
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: {name}
  namespace: {namespace}
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: {target_deployment}
  minReplicas: {min_replicas}
  maxReplicas: {max_replicas}
  metrics:
    # TODO: Add CPU resource metric (target: {cpu_target_percent}%)
    # TODO: Add custom GPU utilization metric (target: {gpu_utilization_target}%)
    # TODO: Add custom queue depth metric (target: {queue_depth_target})
  behavior:
    # TODO: Add scale-up policy (60s stabilization)
    # TODO: Add scale-down policy (300s stabilization)
"""
    return hpa_yaml

print(generate_hpa_config())
print("\n# Complete the TODO sections above!")

In [ ]:
# ASSIGNMENT 2 - SOLUTION

def generate_hpa_config_solution(
    name: str = "vllm-hpa",
    namespace: str = "llm-serving",
    target_deployment: str = "vllm-server",
    min_replicas: int = 1,
    max_replicas: int = 8,
    cpu_target_percent: int = 70,
    gpu_utilization_target: int = 80,
    queue_depth_target: int = 50,
) -> str:
    return f"""\
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: {name}
  namespace: {namespace}
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: {target_deployment}
  minReplicas: {min_replicas}
  maxReplicas: {max_replicas}
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: {cpu_target_percent}
    - type: Pods
      pods:
        metric:
          name: gpu_utilization_percent
        target:
          type: AverageValue
          averageValue: "{gpu_utilization_target}"
    - type: Pods
      pods:
        metric:
          name: vllm_request_queue_depth
        target:
          type: AverageValue
          averageValue: "{queue_depth_target}"
  behavior:
    scaleUp:
      stabilizationWindowSeconds: 60
      policies:
        - type: Pods
          value: 2
          periodSeconds: 60
        - type: Percent
          value: 50
          periodSeconds: 60
      selectPolicy: Max
    scaleDown:
      stabilizationWindowSeconds: 300
      policies:
        - type: Pods
          value: 1
          periodSeconds: 120
      selectPolicy: Min
"""

result = generate_hpa_config_solution()
print(result)
write_config("hpa.yaml", result, subdir="k8s")

### Assignment 3: Write a Deployment Health Check Script

Build a comprehensive deployment verification script that checks all components.

In [ ]:
# ASSIGNMENT 3: Deployment verification script

import time
from dataclasses import dataclass
from enum import Enum

class CheckStatus(Enum):
    PASS = "PASS"
    FAIL = "FAIL"
    WARN = "WARN"
    SKIP = "SKIP"

@dataclass
class DeployCheck:
    name: str
    status: CheckStatus
    message: str
    duration_ms: float = 0.0

class DeploymentVerifier:
    """Verify a complete LLM serving deployment."""
    
    def __init__(self, vllm_url: str, prometheus_url: str = None, grafana_url: str = None):
        self.vllm_url = vllm_url
        self.prometheus_url = prometheus_url
        self.grafana_url = grafana_url
        self.checks: list[DeployCheck] = []
    
    def _timed_check(self, name: str, check_fn) -> DeployCheck:
        """Run a check function with timing."""
        start = time.time()
        try:
            status, message = check_fn()
            elapsed = (time.time() - start) * 1000
            check = DeployCheck(name, status, message, elapsed)
        except Exception as e:
            elapsed = (time.time() - start) * 1000
            check = DeployCheck(name, CheckStatus.FAIL, str(e), elapsed)
        self.checks.append(check)
        return check
    
    def check_vllm_health(self) -> DeployCheck:
        """TODO: Check that the vLLM /health endpoint returns 200."""
        def _check():
            import urllib.request
            # TODO: Make HTTP request to self.vllm_url + "/health"
            # Return (CheckStatus.PASS, "message") or (CheckStatus.FAIL, "message")
            pass  # Replace with your implementation
        return self._timed_check("vLLM Health", _check)
    
    def check_model_available(self) -> DeployCheck:
        """TODO: Check that at least one model is loaded."""
        def _check():
            import urllib.request
            # TODO: Check /v1/models endpoint
            pass
        return self._timed_check("Model Available", _check)
    
    def check_inference_works(self) -> DeployCheck:
        """TODO: Send a test inference request and validate the response."""
        def _check():
            import urllib.request
            # TODO: Send a minimal completion request
            # Verify the response has choices with content
            pass
        return self._timed_check("Inference Works", _check)
    
    def check_prometheus(self) -> DeployCheck:
        """TODO: Check that Prometheus can scrape vLLM metrics."""
        def _check():
            if not self.prometheus_url:
                return (CheckStatus.SKIP, "Prometheus URL not configured")
            # TODO: Query Prometheus API to check vLLM target is up
            pass
        return self._timed_check("Prometheus", _check)
    
    def check_metrics_endpoint(self) -> DeployCheck:
        """TODO: Check that /metrics returns Prometheus-format metrics."""
        def _check():
            import urllib.request
            # TODO: Fetch /metrics and verify it contains expected metric names
            pass
        return self._timed_check("Metrics Endpoint", _check)
    
    def run_all(self) -> list[DeployCheck]:
        """Run all checks and print a report."""
        self.checks = []
        self.check_vllm_health()
        self.check_model_available()
        self.check_metrics_endpoint()
        self.check_inference_works()
        self.check_prometheus()
        
        print(f"\n{'='*60}")
        print(f"Deployment Verification Report")
        print(f"{'='*60}")
        for c in self.checks:
            icon = {"PASS": "+", "FAIL": "X", "WARN": "!", "SKIP": "-"}[c.status.value]
            print(f"  [{icon}] {c.name:20s} | {c.duration_ms:7.1f}ms | {c.message}")
        
        passed = sum(1 for c in self.checks if c.status == CheckStatus.PASS)
        total = sum(1 for c in self.checks if c.status != CheckStatus.SKIP)
        print(f"\n  Result: {passed}/{total} checks passed")
        return self.checks

# Usage example
verifier = DeploymentVerifier(
    vllm_url="http://localhost:8000",
    prometheus_url="http://localhost:9090",
)
print("DeploymentVerifier created.")
print("Complete the TODO methods, then run:")
print("  verifier.run_all()")

In [ ]:
# ASSIGNMENT 3 - SOLUTION

class DeploymentVerifierSolution:
    """Complete deployment verifier."""
    
    def __init__(self, vllm_url: str, prometheus_url: str = None):
        self.vllm_url = vllm_url.rstrip("/")
        self.prometheus_url = prometheus_url.rstrip("/") if prometheus_url else None
        self.checks: list[DeployCheck] = []
    
    def _timed_check(self, name, check_fn):
        start = time.time()
        try:
            status, message = check_fn()
            elapsed = (time.time() - start) * 1000
            check = DeployCheck(name, status, message, elapsed)
        except Exception as e:
            elapsed = (time.time() - start) * 1000
            check = DeployCheck(name, CheckStatus.FAIL, str(e), elapsed)
        self.checks.append(check)
        return check
    
    def check_vllm_health(self):
        def _check():
            import urllib.request
            req = urllib.request.Request(f"{self.vllm_url}/health")
            with urllib.request.urlopen(req, timeout=10) as resp:
                if resp.status == 200:
                    return (CheckStatus.PASS, f"HTTP {resp.status}")
                return (CheckStatus.FAIL, f"HTTP {resp.status}")
        return self._timed_check("vLLM Health", _check)
    
    def check_model_available(self):
        def _check():
            import urllib.request
            req = urllib.request.Request(f"{self.vllm_url}/v1/models")
            with urllib.request.urlopen(req, timeout=10) as resp:
                data = json.loads(resp.read())
                models = [m["id"] for m in data.get("data", [])]
                if models:
                    return (CheckStatus.PASS, f"Models: {', '.join(models)}")
                return (CheckStatus.FAIL, "No models loaded")
        return self._timed_check("Model Available", _check)
    
    def check_inference_works(self):
        def _check():
            import urllib.request
            payload = json.dumps({
                "model": "default",
                "messages": [{"role": "user", "content": "Say OK"}],
                "max_tokens": 3, "temperature": 0
            }).encode()
            req = urllib.request.Request(
                f"{self.vllm_url}/v1/chat/completions",
                data=payload,
                headers={"Content-Type": "application/json"}
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read())
                content = data["choices"][0]["message"]["content"]
                return (CheckStatus.PASS, f"Response: '{content[:50]}'")
        return self._timed_check("Inference Works", _check)
    
    def check_metrics_endpoint(self):
        def _check():
            import urllib.request
            req = urllib.request.Request(f"{self.vllm_url}/metrics")
            with urllib.request.urlopen(req, timeout=10) as resp:
                text = resp.read().decode()
                expected = ["vllm:num_requests", "vllm:gpu_cache"]
                found = [m for m in expected if m in text]
                if found:
                    return (CheckStatus.PASS, f"Found {len(found)} expected metrics")
                return (CheckStatus.WARN, "Metrics endpoint works but missing expected metrics")
        return self._timed_check("Metrics Endpoint", _check)
    
    def check_prometheus(self):
        def _check():
            if not self.prometheus_url:
                return (CheckStatus.SKIP, "Not configured")
            import urllib.request
            req = urllib.request.Request(f"{self.prometheus_url}/api/v1/targets")
            with urllib.request.urlopen(req, timeout=10) as resp:
                data = json.loads(resp.read())
                targets = data.get("data", {}).get("activeTargets", [])
                vllm_targets = [t for t in targets if "vllm" in t.get("labels", {}).get("job", "")]
                if vllm_targets:
                    health = vllm_targets[0].get("health", "unknown")
                    return (CheckStatus.PASS if health == "up" else CheckStatus.WARN,
                            f"vLLM target health: {health}")
                return (CheckStatus.FAIL, "No vLLM scrape target found")
        return self._timed_check("Prometheus", _check)
    
    def run_all(self):
        self.checks = []
        self.check_vllm_health()
        self.check_model_available()
        self.check_metrics_endpoint()
        self.check_inference_works()
        self.check_prometheus()
        
        print(f"\n{'='*60}")
        print(f"Deployment Verification Report")
        print(f"{'='*60}")
        for c in self.checks:
            icon = {"PASS": "+", "FAIL": "X", "WARN": "!", "SKIP": "-"}[c.status.value]
            print(f"  [{icon}] {c.name:20s} | {c.duration_ms:7.1f}ms | {c.message}")
        passed = sum(1 for c in self.checks if c.status == CheckStatus.PASS)
        total = sum(1 for c in self.checks if c.status != CheckStatus.SKIP)
        print(f"\n  Result: {passed}/{total} checks passed")
        return self.checks

# Demo (will show FAIL without running server, which is expected)
verifier = DeploymentVerifierSolution("http://localhost:8000", "http://localhost:9090")
print("Solution verifier ready. Call verifier.run_all() with a running server.")
print("\nExpected output with running server:")
print("  [+] vLLM Health         |   5.2ms | HTTP 200")
print("  [+] Model Available     |  12.1ms | Models: meta-llama/Llama-3.1-8B-Instruct")
print("  [+] Metrics Endpoint    |   8.4ms | Found 2 expected metrics")
print("  [+] Inference Works     | 842.3ms | Response: 'OK'")
print("  [+] Prometheus          |  15.7ms | vLLM target health: up")

## 9. Summary

In this notebook you have:

1. **Built a production vLLM Docker image** with multi-stage builds and security hardening
2. **Created a docker-compose stack** with Prometheus and Grafana monitoring
3. **Written Kubernetes manifests** with GPU scheduling, probes, and PDB
4. **Set up bare-metal systemd services** with proper security and restart policies
5. **Configured SGLang deployments** across all platforms
6. **Built health check tooling** for deployment verification

All configuration files have been written to `./deployment_configs/` and can be used directly.

### Key Takeaways
- Always use health/readiness/startup probes in production
- Mount `/dev/shm` with sufficient size for NCCL communication
- Use PersistentVolumeClaims for model caching across restarts
- Set GPU memory utilization to 0.85-0.9 to leave headroom
- Use PodDisruptionBudgets to prevent full outages during upgrades

In [ ]:
# Final: list all generated config files
print("Generated deployment config files:")
print("=" * 50)
for p in sorted(DEPLOY_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size
        print(f"  {p.relative_to(DEPLOY_DIR)} ({size} bytes)")